## OPERATIONS INTELLIGENCE USING RAG 
       by PRANATHI KETHINEEDI

In [1]:
import pandas as pd

df = pd.read_csv("customer_support_tickets_200k.csv")


In [2]:
df.shape

(200000, 30)

In [3]:
df.isna().sum().sort_values(ascending=False)

browser                        40023
ticket_id                          0
customer_name                      0
issue_complexity_score             0
preferred_contact_time             0
language                           0
payment_method                     0
operating_system                   0
sla_breached                       0
escalated                          0
ticket_resolved_date               0
ticket_created_date                0
resolution_time_hours              0
first_response_time_hours          0
customer_satisfaction_score        0
previous_tickets                   0
customer_tenure_months             0
subscription_type                  0
customer_gender                    0
customer_age                       0
region                             0
channel                            0
status                             0
priority                           0
resolution_notes                   0
issue_description                  0
category                           0
p

In [4]:
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
missing_percent

browser                        20.0115
ticket_id                       0.0000
customer_name                   0.0000
issue_complexity_score          0.0000
preferred_contact_time          0.0000
language                        0.0000
payment_method                  0.0000
operating_system                0.0000
sla_breached                    0.0000
escalated                       0.0000
ticket_resolved_date            0.0000
ticket_created_date             0.0000
resolution_time_hours           0.0000
first_response_time_hours       0.0000
customer_satisfaction_score     0.0000
previous_tickets                0.0000
customer_tenure_months          0.0000
subscription_type               0.0000
customer_gender                 0.0000
customer_age                    0.0000
region                          0.0000
channel                         0.0000
status                          0.0000
priority                        0.0000
resolution_notes                0.0000
issue_description        

In [5]:
df["browser"] = df["browser"].fillna("Unknown")

In [6]:
df_sample = df.sample(n=10000, random_state=42)
df_sample.to_csv("support_tickets_sample_10k.csv", index=False)

In [7]:
df_sample.to_csv

<bound method NDFrame.to_csv of         ticket_id     customer_name                   customer_email  \
119737     119738       Susan Lopez         susan.lopez282@yahoo.com   
72272       72273     Karen Jackson      karen.jackson501@icloud.com   
158154     158155   Elizabeth Brown    elizabeth.brown681@icloud.com   
65426       65427  Richard Anderson   richard.anderson419@icloud.com   
30074       30075    Sarah Anderson       sarah.anderson82@yahoo.com   
...           ...               ...                              ...   
94312       94313  Richard Gonzalez  richard.gonzalez702@outlook.com   
62671       62672        Mary Lopez        mary.lopez623@hotmail.com   
62388       62389    Susan Anderson    susan.anderson384@outlook.com   
8613         8614     John Williams     john.williams824@outlook.com   
162886     162887   Jennifer Garcia   jennifer.garcia327@company.com   

                     product            category  \
119737  Subscription Service          Bug Report   

# creating  searchable text field for each ticket

In [8]:
#converting columns to searchable text field (for use in RAG)
df.columns.tolist()

['ticket_id',
 'customer_name',
 'customer_email',
 'product',
 'category',
 'issue_description',
 'resolution_notes',
 'priority',
 'status',
 'channel',
 'region',
 'customer_age',
 'customer_gender',
 'subscription_type',
 'customer_tenure_months',
 'previous_tickets',
 'customer_satisfaction_score',
 'first_response_time_hours',
 'resolution_time_hours',
 'ticket_created_date',
 'ticket_resolved_date',
 'escalated',
 'sla_breached',
 'operating_system',
 'browser',
 'payment_method',
 'language',
 'preferred_contact_time',
 'issue_complexity_score',
 'customer_segment']

In [9]:
df_sample = df.sample(n=10000, random_state=42).copy()

In [10]:
# creating one text document per ticket
def make_ticket_text(row):
    return f"""
Ticket ID: {row.get('ticket_id', 'Unknown')}
Product: {row.get('product', 'Unknown')}
Category: {row.get('category', 'Unknown')}
Priority: {row.get('priority', 'Unknown')}
Status: {row.get('status', 'Unknown')}
Channel: {row.get('channel', 'Unknown')}
Region: {row.get('region', 'Unknown')}
Browser: {row.get('browser', 'Unknown')}
Escalated: {row.get('escalated', 'Unknown')}
SLA Breached: {row.get('sla_breached', 'Unknown')}
First Response Time Hours: {row.get('first_response_time_hours', 'Unknown')}
Resolution Time Hours: {row.get('resolution_time_hours', 'Unknown')}
Customer Satisfaction Score: {row.get('customer_satisfaction_score', 'Unknown')}
Issue Description: {row.get('issue_description', 'Unknown')}
Resolution Notes: {row.get('resolution_notes', 'Unknown')}
""".strip()

df_sample["rag_text"] = df_sample.apply(make_ticket_text, axis=1)

In [11]:
#check one created doc
print(df_sample["rag_text"].iloc[0])

Ticket ID: 119738
Product: Subscription Service
Category: Bug Report
Priority: High
Status: Open
Channel: Phone
Region: Asia
Browser: Chrome
Escalated: Yes
SLA Breached: No
First Response Time Hours: 57.88
Resolution Time Hours: 205.98
Customer Satisfaction Score: 4
Issue Description: My subscription was cancelled without my request and I need clarification.
Resolution Notes: The duplicate charge was reversed and refund processed within 3-5 business days.


In [12]:
# saving the cleaned sample to a csv file
df_sample.to_csv("support_tickets_rag_sample_10k.csv", index=False)

# Embedding & FAISS indexing

In [13]:
!pip install sentence-transformers faiss-cpu

Defaulting to user installation because normal site-packages is not writeable


In [14]:
import pandas as pd

df_rag = pd.read_csv("support_tickets_rag_sample_10k.csv")
df_rag.shape

(10000, 31)

In [15]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Defaulting to user installation because normal site-packages is not writeable


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
#creating embeddings
texts = df_rag["rag_text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

(10000, 384)

In [17]:
#building faiss index
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))

index.ntotal

10000

In [18]:
#saving faiss index and meta data
import pickle

faiss.write_index(index, "support_ticket_faiss.index")

metadata = df_rag[
    [
        "ticket_id",
        "product",
        "category",
        "priority",
        "status",
        "channel",
        "region",
        "browser",
        "escalated",
        "sla_breached",
        "customer_satisfaction_score",
        "first_response_time_hours",
        "resolution_time_hours",
        "rag_text"
    ]
].to_dict(orient="records")

with open("support_ticket_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [19]:
#testing semantic search
import numpy as np

query = "Why are urgent tickets breaching SLA?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_embedding, 5)

for score, idx in zip(scores[0], indices[0]):
    print("Score:", round(float(score), 4))
    print(metadata[idx]["rag_text"][:800])
    print("-" * 80)

Score: 0.6279
Ticket ID: 102358
Product: Payment Gateway
Category: Login Issue
Priority: Urgent
Status: Pending Customer
Channel: Social Media
Region: North America
Browser: Unknown
Escalated: Yes
SLA Breached: No
First Response Time Hours: 59.36
Resolution Time Hours: 122.35
Customer Satisfaction Score: 3
Issue Description: The system is not syncing data across devices properly.
Resolution Notes: Subscription status corrected and confirmation email sent to customer.
--------------------------------------------------------------------------------
Score: 0.6259
Ticket ID: 130885
Product: Subscription Service
Category: Login Issue
Priority: Urgent
Status: Resolved
Channel: Email
Region: Europe
Browser: Unknown
Escalated: Yes
SLA Breached: Yes
First Response Time Hours: 18.57
Resolution Time Hours: 3.6
Customer Satisfaction Score: 5
Issue Description: There seems to be a discrepancy in my billing statement for this month.
Resolution Notes: Security settings updated and customer notified o

In [20]:
#create summary answer from retrieved tickets
def generate_operations_summary(results):
    total = len(results)

    sla_breached_count = (results["sla_breached"] == "Yes").sum()
    escalated_count = (results["escalated"] == "Yes").sum()

    avg_first_response = results["first_response_time_hours"].mean()
    avg_resolution = results["resolution_time_hours"].mean()
    avg_satisfaction = results["customer_satisfaction_score"].mean()

    top_products = results["product"].value_counts().head(3)
    top_categories = results["category"].value_counts().head(3)
    top_channels = results["channel"].value_counts().head(3)
    top_regions = results["region"].value_counts().head(3)

    print("OPERATIONS INTELLIGENCE SUMMARY")
    print("-" * 50)

    print("\n1. Key Issue")
    print(
        f"Among the retrieved tickets, {sla_breached_count} out of {total} tickets breached SLA, "
        f"and {escalated_count} were escalated. The tickets are mostly concentrated around "
        f"{', '.join(top_categories.index.tolist())} issues."
    )

    print("\n2. Likely Root Cause")
    print(
        f"The pattern suggests delays are connected to high operational complexity, long first response times, "
        f"and recurring issues in products such as {', '.join(top_products.index.tolist())}."
    )

    print("\n3. Supporting Evidence")
    print(f"Average first response time: {avg_first_response:.2f} hours")
    print(f"Average resolution time: {avg_resolution:.2f} hours")
    print(f"Average satisfaction score: {avg_satisfaction:.2f}")
    print(f"Top products: {top_products.to_dict()}")
    print(f"Top categories: {top_categories.to_dict()}")
    print(f"Top channels: {top_channels.to_dict()}")
    print(f"Top regions: {top_regions.to_dict()}")

    print("\n4. Recommended Action")
    print(
        "Prioritize SLA-breached and escalated tickets by product and category, create faster routing rules "
        "for recurring login, billing, and payment issues, and monitor first response time as an early warning KPI."
    )

    print("\n5. Limitations")
    print(
        "This summary is based only on the top retrieved tickets for the query, not the full dataset. "
        "It should be validated with aggregate analysis across all tickets."
    )

In [21]:
import pandas as pd
import numpy as np

# Load your prepared RAG sample
df_rag = pd.read_csv("support_tickets_rag_sample_10k.csv")

# If these numeric columns loaded as text, convert them
numeric_cols = [
    "customer_satisfaction_score",
    "first_response_time_hours",
    "resolution_time_hours"
]

for col in numeric_cols:
    if col in df_rag.columns:
        df_rag[col] = pd.to_numeric(df_rag[col], errors="coerce")

def search_tickets(query, top_k=10):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    rows = []
    for score, idx in zip(scores[0], indices[0]):
        row = df_rag.iloc[idx].copy()
        row["similarity_score"] = round(float(score), 4)
        rows.append(row)

    return pd.DataFrame(rows)

# Test
results = search_tickets("Why are urgent tickets breaching SLA?", top_k=10)
results[
    [
        "similarity_score",
        "ticket_id",
        "product",
        "category",
        "priority",
        "status",
        "channel",
        "region",
        "sla_breached",
        "escalated",
        "customer_satisfaction_score",
        "first_response_time_hours",
        "resolution_time_hours"
    ]
]

,similarity_score,ticket_id,product,category,priority,status,channel,region,sla_breached,escalated,customer_satisfaction_score,first_response_time_hours,resolution_time_hours
6563,0.6279,102358,Payment Gateway,Login Issue,Urgent,Pending Customer,Social Media,North America,No,Yes,3,59.36,122.35
8807,0.6259,130885,Subscription Service,Login Issue,Urgent,Resolved,Email,Europe,Yes,Yes,5,18.57,3.60
6823,0.6175,52667,Payment Gateway,Subscription Cancellation,Low,Closed,Email,Asia,Yes,Yes,2,4.34,112.93
9640,0.6172,16151,Subscription Service,Account Suspension,Medium,In Progress,Social Media,Asia,No,Yes,2,62.11,35.27
1469,0.6161,71085,Web Portal,Login Issue,High,Pending Customer,Email,Australia,Yes,Yes,4,36.87,196.73
1498,0.6160,178683,Payment Gateway,Security Concern,Medium,Pending Customer,Web Form,Europe,No,Yes,5,52.35,83.95
488,0.6160,161796,Subscription Service,Account Suspension,Urgent,Pending Customer,Email,South America,Yes,Yes,1,18.31,100.03
332,0.6150,51822,Subscription Service,Feature Request,Urgent,Closed,Phone,Asia,No,Yes,2,67.02,63.50
4117,0.6144,1966,Subscription Service,Security Concern,Low,Closed,Social Media,Europe,Yes,Yes,4,38.68,184.61
6151,0.6136,146036,Subscription Service,Security Concern,Urgent,Pending Customer,Phone,Australia,Yes,Yes,5,61.43,11.08


In [22]:
results = search_tickets("Why are urgent tickets breaching SLA?", top_k=10)
generate_operations_summary(results)

OPERATIONS INTELLIGENCE SUMMARY
--------------------------------------------------

1. Key Issue
Among the retrieved tickets, 6 out of 10 tickets breached SLA, and 10 were escalated. The tickets are mostly concentrated around Login Issue, Security Concern, Account Suspension issues.

2. Likely Root Cause
The pattern suggests delays are connected to high operational complexity, long first response times, and recurring issues in products such as Subscription Service, Payment Gateway, Web Portal.

3. Supporting Evidence
Average first response time: 41.90 hours
Average resolution time: 91.41 hours
Average satisfaction score: 3.30
Top products: {'Subscription Service': 6, 'Payment Gateway': 3, 'Web Portal': 1}
Top categories: {'Login Issue': 3, 'Security Concern': 3, 'Account Suspension': 2}
Top channels: {'Email': 4, 'Social Media': 3, 'Phone': 2}
Top regions: {'Europe': 3, 'Asia': 3, 'Australia': 2}

4. Recommended Action
Prioritize SLA-breached and escalated tickets by product and catego

In [23]:
results = search_tickets("What are major red flags that affect customer satisfaction?", top_k=10)
generate_operations_summary(results)

OPERATIONS INTELLIGENCE SUMMARY
--------------------------------------------------

1. Key Issue
Among the retrieved tickets, 6 out of 10 tickets breached SLA, and 2 were escalated. The tickets are mostly concentrated around Feature Request, Performance Issue, Payment Problem issues.

2. Likely Root Cause
The pattern suggests delays are connected to high operational complexity, long first response times, and recurring issues in products such as E-commerce Store, CRM Platform, Billing System.

3. Supporting Evidence
Average first response time: 35.77 hours
Average resolution time: 80.77 hours
Average satisfaction score: 3.20
Top products: {'E-commerce Store': 6, 'CRM Platform': 3, 'Billing System': 1}
Top categories: {'Feature Request': 4, 'Performance Issue': 2, 'Payment Problem': 1}
Top channels: {'Social Media': 3, 'Email': 2, 'Chat': 2}
Top regions: {'Europe': 3, 'North America': 2, 'Asia': 2}

4. Recommended Action
Prioritize SLA-breached and escalated tickets by product and catego